In [ ]:
using CairoMakie, NBInclude, Random, Statistics, UnPack, FFTW, LaTeXStrings
@nbinclude("../02_ETD_solvers/etd_euler.ipynb")
@nbinclude("../02_ETD_solvers/etd_rk2.ipynb")
@nbinclude("../02_ETD_solvers/etd_rk3.ipynb")
@nbinclude("../02_ETD_solvers/etd_rk4.ipynb")
@nbinclude("../set_makie_defaults.ipynb")
@nbinclude("../05_RD_utils/rd_visualization.ipynb")

In [ ]:
function rd_solve_one_species(;f, D, u0_fun, ℓ::Float64, tspan::Tuple, N::Int, Δt::Float64, p = nothing)
    """
    Solves 1D reaction-diffusion PDEs with *periodic* boundary conditions 
    
                    u_t = Du_{xx} + f(u,x,t,p)    0 ≤ x ≤ ℓ, tspan[1] ≤ t ≤ tspan[2] 
                    u(x,tspan[1]) = u0_fun.(0)
                    u(0,t) = u(ℓ,t)  and u_x(0,t) = u_x(ℓ,t) 

    The PDE is "diagonalized" via the Fourier transform, and a system of N ODEs is formulated for the 
    discrete Fourier coefficients with indices -N/2,...,N/2 - 1, which is then solved using the 
    Exponential Time Differencing 4th order Runge-Kutta (ETD-RK4) method. 
        
    PARAMETERS
    ----------
    f :: a function of the form f(u,x,t,p), where p is a parameter 
    D :: diffusion coefficient (a positive scalar) 
    u0_fun :: initial condition function of the form u0_fun(x), where x is a scalar
    ℓ :: length of the periodic domain 
    tspan :: the time interval over which to integrate
    Δt :: the step size (fixed)
    p :: parameter for the reaction term `f` 
    """

    x = collect(range(0, step = ℓ/N, length = N))    #spatial discretization 0 = x_0 < ... < x_{N-1}  
    U0 = u0_fun.(x)
    U0_hat = fft(U0)    #Get the DFT of U0 
    
    #Assume for now that N = the number of Fourier coefficients, so that each solution iterate U^n is a vector of length N. 

    #Frequency indices 
    K = vcat(0:Int(floor(N/2)), -(Int(ceil(N/2))-1):-1)

    #N eigenvalues of the operator D∂_xx 
    λ = @. -D*(2*π*K/ℓ)^2  

    #Diagonal matrix 
    Λ = Diagonal(λ)

    function G(U_hat, p, t)
        U = real.(ifft(U_hat))
        F = f.(U,x,t,Ref(p))      #broadcast over indices of U and x [F[j] = f(U[j],x[j],t,p) for j=1,…,N] 
        return fft(F)
    end 

    # @time sol_hat = arnoldi == true ? etd_rk4_arnoldi(Λ, G, U0_hat; Δt, tspan, p = p) : etd_rk4(Λ, G, U0_hat; Δt, tspan, p = p)

    U_vals = real.(ifft.(sol_hat.u))
    return (u = U_vals, t = sol_hat.t, x = x, Δt = Δt)
end 

In [ ]:
function rd_solve_two_species(; f, g, D_u, D_v, u0_fun, v0_fun, ℓ::Float64, tspan::Tuple, Δt::Float64, N::Int, 
                                p = nothing, solver = etd_euler, ϵ::Float64 = 1e-3)
    """
    Solves the two-species reaction-diffusion system

        u_t = D_u u_xx + f(u,v,x,t,p)
        v_t = D_v v_xx + g(u,v,x,t,p)

    on [0,ℓ] with periodic boundary conditions.
    """

    # Spatial grid
    x = collect(range(0, step=ℓ/N, length=N))

    #Initial conditions in physical space
    U0 = u0_fun.(x)
    V0 = v0_fun.(x)

    #Initial conditions in Fourier space
    U0_hat = fft(U0)
    V0_hat = fft(V0)

    #Combined Fourier-space initial condition
    Y0_hat = vcat(U0_hat, V0_hat)

    # Frequency indices in FFT order
    #K = [k <= N÷2 ? k : k - N for k in 0:N-1]
    K = vcat(0:Int(floor(N/2)), -(Int(ceil(N/2))-1):-1)

    # Eigenvalues for D_u ∂xx and D_v ∂xx
    λ_u = @. -D_u * (2π*K/ℓ)^2
    λ_v = @. -D_v * (2π*K/ℓ)^2

    #Diagonal matrix of eigenvalues 
    Λ = Diagonal(vcat(λ_u, λ_v))

    function G(Y_hat, p, t)

        #Get the Fourier coeffs for each species 
        U_hat = Y_hat[1:N]
        V_hat = Y_hat[N+1:2N]

        # Transform back to physical space
        U = real.(ifft(U_hat))
        V = real.(ifft(V_hat))

        # Evaluate nonlinearities pointwise
        F = f.(U, V, x, t, Ref(p))
        H = g.(U, V, x, t, Ref(p))

        # Transform nonlinearities back to Fourier space
        F_hat = fft(F)
        H_hat = fft(H)

        return vcat(F_hat, H_hat)
    end

    sol_hat = solver(Λ, G, Y0_hat; tspan = tspan, Δt = Δt, p = p, ϵ = ϵ)
    #sol_hat = arnoldi == true ? etd_rk4_arnoldi(Λ, G, Y0_hat; Δt=Δt, tspan=tspan, p=p) : etd_rk4(Λ, G, Y0_hat; Δt=Δt, tspan=tspan, p=p)
    
    # Convert solution back to physical space
    U_vals = [real.(ifft(Y_hat[1:N])) for Y_hat in sol_hat.u]
    V_vals = [real.(ifft(Y_hat[N+1:2N])) for Y_hat in sol_hat.u]

    return (u = U_vals, v = V_vals, t = sol_hat.t, x = x, Δt = Δt)
end

In [ ]:
# xs = 0:0.01:1
# foo(x) = abs(x-0.7) < 0.1 || abs(x-0.3) < 0.1 ? 1.0 : 0.2

# lines(xs, foo.(xs))

In [ ]:
# # Reaction term from the model
# reaction(u, v, p) = (p.b0 + p.γ * u^2 / (p.K^2 + u^2)) * v - p.δ * u

# # Spatial forcing g(x)
# #forcing(x, p) = p.ε * cos(2π * x / p.ℓ)   # modify as desired

# forcing(x,p) = zero(x)

# # PDE after dividing both equations by ε:
# # u_t = ε u_xx + (1/ε) f(u,v) + g(x)
# # v_t = (1/ε) v_xx - (1/ε) f(u,v) - g(x)

# function F_u(u, v, x, t, p)
#     return (1 / p.ε) * reaction(u, v, p) + forcing(x, p)
# end

# function F_v(u, v, x, t, p)
#     return -(1 / p.ε) * reaction(u, v, p) - forcing(x, p)
# end

# L = ℓ = 1.0

# #1. Step function
# u0_fun1(x) = abs(x-ℓ/2) ≤ ℓ/4 ? 1.0 : 0.0
# v0_fun1(x) = 0.8

# # 2. Out-of-phase sinusoidal waves
# u0_fun2(x) = 1.0 - 0.1*cos(2π*x/ℓ)
# v0_fun2(x) = 0.2 + 0.1*cos(2π*x/ℓ)

# # 3. Higher-frequency perturbations
# u0_fun3(x) = 1.0 + 0.05*cos(4π*x/ℓ)
# v0_fun3(x) = 0.5 + 0.05*sin(6π*x/ℓ)

# # 4. Mixed Fourier modes
# u0_fun4(x) = 1.0 + 0.08*cos(2π*x/ℓ) + 0.03*sin(6π*x/ℓ)
# v0_fun4(x) = 0.5 + 0.06*sin(4π*x/ℓ) - 0.02*cos(8π*x/ℓ)

# # 5. \elloca\ellized periodic bump
# u0_fun5(x) = 1.0 + 0.4*exp(-20*(sin(π*x/ℓ))^2)
# v0_fun5(x) = 0.5

# # 6. Two step functions 
# u0_fun6(x) = abs(x-0.7) < 0.1 || abs(x-0.3) < 0.1 ? 1.0 : 0.2
# v0_fun6(x) = 0.2

# # # 6. Two \elloca\ellized periodic bumps, shifted
# # u0_fun6(x) = 1.0 + 0.4*exp(-30*(sin(π*x/ℓ))^2)
# # v0_fun6(x) = 0.5 + 0.3*exp(-30*(sin(π*(x - ℓ/2)/ℓ))^2)

# # 7. Step-\ellike periodic transition
# u0_fun7(x) = 1.0 + 0.3*tanh(10*cos(2π*x/ℓ))
# v0_fun7(x) = 0.5 - 0.3*tanh(10*cos(2π*x/ℓ))

# # 8. Random-\ellooking smooth perturbation
# u0_fun8(x) = 1.0 + 0.04*cos(2π*x/ℓ) + 0.03*cos(6π*x/ℓ) + 0.02*sin(10π*x/ℓ)
# v0_fun8(x) = 0.5 + 0.05*sin(4π*x/ℓ) - 0.02*cos(12π*x/ℓ)

# # 9. Near-homogeneous initial data
# u0_fun9(x) = 1.0 + 1e-3*cos(2π*x/ℓ)
# v0_fun9(x) = 0.5 + 1e-3*sin(2π*x/ℓ)

# # 10. Phase-shifted pattern
# u0_fun10(x) = 1.0 + 0.1*cos(2π*x/ℓ)
# v0_fun10(x) = 0.5 + 0.1*cos(2π*(x - ℓ/4)/ℓ)

# u0_funcs = [u0_fun1, u0_fun2, u0_fun3, u0_fun4, u0_fun5, u0_fun6, u0_fun7, u0_fun8, u0_fun9, u0_fun10]
# v0_funcs = [v0_fun1, v0_fun2, v0_fun3, v0_fun4, v0_fun5, v0_fun6, v0_fun7, v0_fun8, v0_fun9, v0_fun10]

# # Parameters
# p = (
#     ε = 0.05,    #sqrt of ratio of D_u/D_v; small ϵ ⟹ D_u << D_v, i.e. membrane diffusion << cytosol diffusion
#     b0 = 0.01,   #basal activation rate 
#     γ = 1.0,     
#     K = 0.5,   
#     δ = 1.0,    #Increasing from 0.2 --> 1.0 leads to wave pinning! ***Need parameter values that make f bistable!!***
#     A = 0.1,
#     ℓ = 1.0)

In [ ]:
# function plot_reaction_curves(p; vvals = [0.6, 0.8, 1.0, 1.2, 1.4])
    
#     fig = Figure(size = (500, 400))
#     ax = Axis(fig[1, 1], xlabel = L"u", ylabel = L"f(u,v)", title = "Reaction curves")

#     uvals = range(0, 2, length = 500)

#     for v in vvals
#         fvals = [reaction(u, v, p) for u in uvals]
#         lines!(ax, uvals, fvals, label = L"v = %$v")
#     end

#     hlines!(ax, [0], linestyle = :dash)

#     Legend(fig[1,2], ax)
#     display(fig)
# end

# plot_reaction_curves(p)

In [ ]:
# # Domain/time parameters
# ℓ = p.ℓ
# N = 256
# Δt = 1e-2
# L = ℓ

# # Initial conditions
# for j=1:6
    
#     tf = j != 6 ? 5.0 : 0.1
#     tspan = (0.0, tf)
 
#     # Solve
#     sol = rd_solve_two_species(f = F_u, g = F_v, D_u = p.ε^2, D_v = 1.0, u0_fun = u0_funcs[j], v0_fun = v0_funcs[j],
#                                ℓ = ℓ, tspan = tspan, N = N, Δt = Δt, p = p, ϵ = 1.0, solver = etd_rk4);
    
#     U = sol.u
#     V = sol.v
#     t = sol.t
#     x = sol.x;

  

#     fig = plot_profiles(sol; plot_times = [0, tf/8, tf/4, tf], overlay = true, axislabelsize = 15, titlesize = 15)
#     display(fig)
# end 